# 06 · Expanded engine validation

**Implementation in progress: 140 of 1,185 card effects, with all 11 base hero powers written. Full Standard is not ready.**

This notebook is prepared for you to run. The assistant has **not executed its cells, rule fixtures, matches, or training**.

**Run → Run All Cells** runs targeted rule fixtures and displays the current implementation gaps. Fixtures set up small game states and check exact outcomes. This notebook does not run a match batch or train the old model on the larger card book.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "expanded" / "game.py").exists()), None)
if ROOT is None: raise RuntimeError("Open this notebook inside death-knight-lab.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from expanded.status import readiness, run_rule_fixtures
state = readiness()
print("Card effects written:", state["effects_written"], "/", state["catalog_cards"])
print("Base hero powers written:", len(state["base_hero_powers_written"]))
print("Full Standard ready:", state["full_standard_ready"])

## Validate the new rules
These fixtures cover base hero powers for every class; class/rune restrictions; hidden information; Overload; spell damage; Stealth and Elusive targeting; Freeze timing; Outcast; Combo; Lifesteal; generated cards; deathrattles; and restoring the state if an effect fails. Passing them supports only these tested cases. It does not certify every card interaction.

In [ ]:
validation = run_rule_fixtures()
print("Passed:", validation["success"])
print("Checks run:", validation["tests_run"])
if not validation["success"]:
    raise RuntimeError("Rule checks failed. Keep this output for diagnosis; no training has started.")

## Exact remaining cards
The missing-card report contains IDs, classes, names, and rules text for every unimplemented catalog card. No unknown card is silently played with its effect omitted.

In [ ]:
state = readiness()
folder = ROOT / "runs" / "expanded_validation"
folder.mkdir(parents=True, exist_ok=True)
path = folder / "readiness.json"
temporary = path.with_suffix(".tmp")
temporary.write_text(json.dumps(state, indent=2) + "\n")
temporary.replace(path)
print("Saved:", path)
print("Still missing:", len(state["missing_cards"]), "card effects")
print("Full Standard ready:", state["full_standard_ready"])
for task in state["remaining"]: print("•", task)

## What has changed
`expanded.Game` accepts two class-aware decks rather than assuming two Death Knights. It uses explicit card effects and rejects unsupported decks before the first draw. Actions expose legal choices without showing opponents’ hands or shuffled decks. Failed effects restore the state and random stream.

The old `engine` and `learner` packages remain unchanged, so your previous model can still be loaded in notebooks 03 and 04. It cannot be used as a full-Standard model.

Next engine work includes choice flows, persistent triggers and auras, locations, secrets, quests, modern expansion mechanics, complete generation pools, exceptional construction rules, and the remaining individual cards. The catalog alone cannot provide those behaviors. The full-Standard entry point remains blocked until that work and validation are complete.